In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("test").getOrCreate()
spark.range(5).show()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/08 12:48:01 WARN Utils: Your hostname, DESKTOP-7KEFMSR, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/05/08 12:48:01 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/08 12:48:03 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
+---+



In [3]:
BASE_PATH = "file:///home/FA23-BDS-021/ecommerce_data/"

customers = spark.read.csv(BASE_PATH + "customers.csv", header=True, inferSchema=True)
products  = spark.read.csv(BASE_PATH + "products.csv", header=True, inferSchema=True)
sessions  = spark.read.csv(BASE_PATH + "sessions.csv", header=True, inferSchema=True)
events    = spark.read.csv(BASE_PATH + "events.csv", header=True, inferSchema=True)
orders    = spark.read.csv(BASE_PATH + "orders.csv", header=True, inferSchema=True)
reviews   = spark.read.csv(BASE_PATH + "reviews.csv", header=True, inferSchema=True)

In [4]:
events.printSchema()
sessions.printSchema()

root
 |-- event_id: integer (nullable = true)
 |-- session_id: integer (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- event_type: string (nullable = true)
 |-- product_id: double (nullable = true)
 |-- qty: double (nullable = true)
 |-- cart_size: double (nullable = true)
 |-- payment: string (nullable = true)
 |-- discount_pct: double (nullable = true)
 |-- amount_usd: double (nullable = true)

root
 |-- session_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- start_time: timestamp (nullable = true)
 |-- device: string (nullable = true)
 |-- source: string (nullable = true)
 |-- country: string (nullable = true)



In [5]:
event_session = events.join(sessions, "session_id", "left")

In [6]:
event_session = event_session.withColumnRenamed("customer_id", "user_id")

In [7]:
from pyspark.sql.functions import when, col

event_scored = event_session.withColumn(
    "event_score",
    when(col("event_type") == "page_view", 1)
    .when(col("event_type") == "add_to_cart", 3)
    .when(col("event_type") == "checkout", 7)
    .otherwise(0)
)

In [8]:
event_scored = event_scored.filter(col("product_id").isNotNull())

In [9]:
user_product = event_scored.groupBy("user_id", "product_id") \
    .sum("event_score") \
    .withColumnRenamed("sum(event_score)", "interaction_score")

In [10]:
user_product.show(10)

+-------+----------+-----------------+
|user_id|product_id|interaction_score|
+-------+----------+-----------------+
|   6145|    1012.0|                1|
|  14764|     769.0|                4|
|   3044|    1018.0|                1|
|   6808|     802.0|                4|
|   2494|      31.0|                1|
|   4760|     833.0|                1|
|   6404|     849.0|                4|
|  12725|     961.0|                4|
|   5958|    1176.0|                1|
|  11445|     902.0|                1|
+-------+----------+-----------------+
only showing top 10 rows


In [12]:
order_items = spark.read.csv(BASE_PATH + "order_items.csv", header=True, inferSchema=True)

In [13]:
order_join = orders.join(order_items, "order_id", "left")

In [14]:
from pyspark.sql.functions import col, lit

order_interactions = order_join.select(
    col("customer_id").alias("user_id"),
    col("product_id"),
    lit(10).alias("interaction_score")   # strong purchase signal
)

In [15]:
final_interactions = user_product.union(order_interactions)

In [16]:
final_interactions = final_interactions.groupBy("user_id", "product_id") \
    .sum("interaction_score") \
    .withColumnRenamed("sum(interaction_score)", "interaction_score")

In [17]:
from pyspark.sql.functions import sum as _sum

product_popularity = final_interactions.groupBy("product_id") \
    .agg(_sum("interaction_score").alias("popularity_score"))

In [18]:
top_products = product_popularity.orderBy("popularity_score", ascending=False)
top_products.show(10)

+----------+----------------+
|product_id|popularity_score|
+----------+----------------+
|     496.0|            2883|
|     442.0|            2831|
|     404.0|            2807|
|     769.0|            2684|
|    1148.0|            2649|
|     861.0|            2646|
|    1028.0|            2644|
|     358.0|            2636|
|     231.0|            2628|
|    1039.0|            2624|
+----------+----------------+
only showing top 10 rows


In [19]:
user_history = final_interactions.select("user_id", "product_id")

In [20]:
all_users = final_interactions.select("user_id").distinct()

candidates = all_users.crossJoin(product_popularity)

In [21]:
recommendations = candidates.join(
    user_history,
    on=["user_id", "product_id"],
    how="left_anti"
)

In [22]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

window = Window.partitionBy("user_id").orderBy(col("popularity_score").desc())

ranked_recommendations = recommendations.withColumn(
    "rank",
    row_number().over(window)
)

In [23]:
top_n_recommendations = ranked_recommendations.filter(col("rank") <= 5)

In [24]:
top_n_recommendations.show(20)

+-------+----------+----------------+----+
|user_id|product_id|popularity_score|rank|
+-------+----------+----------------+----+
|     12|     496.0|            2883|   1|
|     12|     442.0|            2831|   2|
|     12|     404.0|            2807|   3|
|     12|     769.0|            2684|   4|
|     12|    1148.0|            2649|   5|
|     26|     496.0|            2883|   1|
|     26|     442.0|            2831|   2|
|     26|     404.0|            2807|   3|
|     26|     769.0|            2684|   4|
|     26|    1148.0|            2649|   5|
|     27|     496.0|            2883|   1|
|     27|     442.0|            2831|   2|
|     27|     404.0|            2807|   3|
|     27|     769.0|            2684|   4|
|     27|    1148.0|            2649|   5|
|     28|     496.0|            2883|   1|
|     28|     442.0|            2831|   2|
|     28|     404.0|            2807|   3|
|     28|     769.0|            2684|   4|
|     28|    1148.0|            2649|   5|
+-------+--

In [25]:
url = "jdbc:postgresql://localhost:5432/ecommerce_db"

properties = {
    "user": "postgres",
    "password": "car.gemera",
    "driver": "org.postgresql.Driver"
}

In [26]:
final_output = top_n_recommendations.select(
    "user_id",
    "product_id",
    "popularity_score",
    "rank"
)

In [ ]:
final_output.write \
    .jdbc(
        url=url,
        table="user_recommendations",
        mode="append",   # replace table if exists
        properties=properties
    )

[648.208s][warning][gc,alloc] Executor task launch worker for task 5.0 in stage 109.0 (TID 195): Retried waiting for GCLocker too often allocating 4194306 words


26/05/08 12:58:46 WARN TaskMemoryManager: Failed to allocate a page (33554432 bytes), try again.


In [28]:
final_interactions.write.jdbc(
    url=url,
    table="user_product_interactions",
    mode="overwrite",
    properties=properties
)

In [29]:
final_interactions.printSchema()

root
 |-- user_id: integer (nullable = true)
 |-- product_id: double (nullable = true)
 |-- interaction_score: long (nullable = true)



In [30]:
from pyspark.sql.functions import col

als_data = final_interactions.select(
    col("user_id").cast("integer"),
    col("product_id").cast("integer"),
    col("interaction_score").cast("float")
)